# Adversarial Machine Learning — FGSM Attack & Defensive Distillation

This notebook demonstrates two concepts on MNIST:

1. **Attack** — Fast Gradient Sign Method (FGSM, Goodfellow et al. 2014)
2. **Defence** — Defensive Distillation (Papernot et al. 2016)

All helper code lives in the companion Python files:

| File | Contents |
|------|----------|
| `config.py` | Hyperparameters |
| `models.py` | `Net` (teacher/baseline), `NetF1` (student) |
| `attacks.py` | `fgsm_attack` |
| `train.py` | `fit`, `test` |
| `defense.py` | `SoftLabelDataset`, `defense` |
| `utils.py` | Plotting helpers |

## 0. Setup

In [ ]:
import numpy as np
import torch
import torch.optim as optim
from torchvision import transforms, datasets

from config  import *
from models  import Net
from train   import fit, test
from defense import defense
from utils   import plot_loss_curves, plot_accuracy_vs_epsilon, plot_adversarial_examples

np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 1. Data

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.0,), (1.0,))   # no-op; keeps values in [0, 1]
])

full_train = datasets.MNIST(root='./data', train=True,  transform=transform, download=True)
test_set   = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_set, val_set = torch.utils.data.random_split(full_train, [50000, 10000])

train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_loader)}  Val: {len(val_loader)}  Test: {len(test_loader)}")

## 2. Baseline model training

In [ ]:
model     = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=LR, betas=BETAS)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=LR_FACTOR, patience=LR_PATIENCE)

train_loss, val_loss = fit(model, device, optimizer, scheduler,
                            train_loader, val_loader, epochs=EPOCHS)

In [ ]:
plot_loss_curves(train_loss, val_loss, title="Baseline model")

## 3. FGSM attack

$$\tilde{x} = x + \epsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$

We evaluate the baseline model at several epsilon values.
Accuracy should drop sharply as epsilon increases.

In [ ]:
accuracies, examples = [], []
for eps in EPSILONS:
    acc, ex = test(model, device, test_loader, eps)
    accuracies.append(acc)
    examples.append(ex)

plot_accuracy_vs_epsilon(EPSILONS, accuracies, title="Baseline — FGSM robustness")
plot_adversarial_examples(EPSILONS, examples)

## 4. Defence — Defensive Distillation

**Key idea** (Papernot et al., 2016):

1. Train a *teacher* model **F** at high temperature **T** — the soft outputs are less peaked, carrying more uncertainty information.
2. Use **F**'s soft probability vectors as training targets for a smaller *student* model **F'**.
3. The student learns a smoother decision surface, reducing the gradient signal available to an attacker.
4. At test time, evaluate with T = 1 (standard argmax).

The `defense()` function handles the full pipeline; see `defense.py` for implementation details and bug-fix notes.

In [ ]:
student, def_accuracies, def_examples = defense(
    device, train_loader, val_loader, test_loader,
    epochs=EPOCHS, temperature=TEMPERATURE, epsilons=EPSILONS
)

### 4.1 Compare baseline vs distilled student

In [ ]:
from utils import plot_defense_comparison

plot_defense_comparison(
    EPSILONS,
    baseline_acc=accuracies,
    defense_acc=def_accuracies,
    defense_label=f"Distilled student (T={TEMPERATURE})"
)